In [1]:
import os
import re
import json
from pathlib import Path
from collections import defaultdict
import random
import numpy as np

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"✓ Random seed set to: {RANDOM_SEED}")
print(f"✓ Imports complete")

✓ Random seed set to: 42
✓ Imports complete


In [2]:
# Base paths
BASE_DIR = Path('/teamspace/studios/this_studio')
DATA_DIR = BASE_DIR / 'TransMuCoRes' / 'processed_data_full'
OUTPUT_DIR = BASE_DIR / 'final_experiments'

# BenCoref domain files
BENCOREF_DOMAINS = ['biography', 'descriptive', 'novel', 'story']
BENCOREF_FILES = {
    domain: DATA_DIR / f'bencoref_{domain}.conll' 
    for domain in BENCOREF_DOMAINS
}

# TransMuCoRes file
TRANSMUCORES_FILE = DATA_DIR / 'transmucores_all_100docs.conll'

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)

# Verify all files exist
print("Checking data files...")
print(f"\nData directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

print("\n" + "="*60)
print("BenCoref domain files:")
print("="*60)
for domain, filepath in BENCOREF_FILES.items():
    exists = "✓" if filepath.exists() else "✗ MISSING"
    print(f"{exists} {domain:12s}: {filepath.name}")

print("\n" + "="*60)
print("TransMuCoRes file:")
print("="*60)
exists = "✓" if TRANSMUCORES_FILE.exists() else "✗ MISSING"
print(f"{exists} {TRANSMUCORES_FILE.name}")

# Check if all files exist
all_files = list(BENCOREF_FILES.values()) + [TRANSMUCORES_FILE]
if all(f.exists() for f in all_files):
    print("\n✓ All data files found! Ready to proceed.")
else:
    print("\n✗ Some files are missing! Please check the paths.")

Checking data files...

Data directory: /teamspace/studios/this_studio/TransMuCoRes/processed_data_full
Output directory: /teamspace/studios/this_studio/final_experiments

BenCoref domain files:
✓ biography   : bencoref_biography.conll
✓ descriptive : bencoref_descriptive.conll
✓ novel       : bencoref_novel.conll
✓ story       : bencoref_story.conll

TransMuCoRes file:
✓ transmucores_all_100docs.conll

✓ All data files found! Ready to proceed.


In [3]:
def extract_documents_from_conll(filepath):
    """
    Extract individual documents from a CoNLL file.
    Documents are separated by #begin document and #end document markers.
    
    Returns:
        list of (doc_id, doc_text) tuples
    """
    documents = []
    current_doc = []
    current_doc_id = None
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            
            if line.startswith('#begin document'):
                # Extract document ID from: #begin document (doc_id); part 000
                match = re.search(r'#begin document \((.+?)\)', line)
                if match:
                    current_doc_id = match.group(1)
                current_doc = [line]
            
            elif line.startswith('#end document'):
                current_doc.append(line)
                if current_doc_id:
                    doc_text = '\n'.join(current_doc)
                    documents.append((current_doc_id, doc_text))
                current_doc = []
                current_doc_id = None
            
            else:
                current_doc.append(line)
    
    return documents

print("✓ Function 'extract_documents_from_conll' defined")

# Quick test on one file
test_file = BENCOREF_FILES['biography']
test_docs = extract_documents_from_conll(test_file)
print(f"\n✓ Test: Extracted {len(test_docs)} documents from biography domain")
print(f"  Sample doc ID: {test_docs[0][0]}")

✓ Function 'extract_documents_from_conll' defined

✓ Test: Extracted 17 documents from biography domain
  Sample doc ID: train_001


In [4]:
# Load all BenCoref documents, organized by domain
bencoref_by_domain = {}

print("Loading BenCoref documents by domain...")
print("="*60)

for domain in BENCOREF_DOMAINS:
    filepath = BENCOREF_FILES[domain]
    docs = extract_documents_from_conll(filepath)
    bencoref_by_domain[domain] = docs
    print(f"{domain:12s}: {len(docs):2d} documents")

# Calculate total
total_bencoref = sum(len(docs) for docs in bencoref_by_domain.values())
print("="*60)
print(f"{'TOTAL':12s}: {total_bencoref:2d} documents")

# Verify we have 122 total
if total_bencoref == 122:
    print("\n✓ Correct! BenCoref has 122 documents total")
else:
    print(f"\n⚠️  Warning: Expected 122 documents, got {total_bencoref}")

Loading BenCoref documents by domain...
biography   : 17 documents
descriptive : 36 documents
novel       : 13 documents
story       : 56 documents
TOTAL       : 122 documents

✓ Correct! BenCoref has 122 documents total


In [5]:
def stratified_split(documents_by_domain, train_size, dev_size, test_size, random_seed=42):
    """
    Perform stratified split across domains.
    Each domain is split proportionally to maintain domain distribution.
    
    Args:
        documents_by_domain: dict of {domain: [(doc_id, doc_text), ...]}
        train_size: total number of docs for training
        dev_size: total number of docs for dev
        test_size: total number of docs for test
        random_seed: random seed for reproducibility
    
    Returns:
        train_docs, dev_docs, test_docs: lists of (doc_id, doc_text, domain)
        split_info: dict with detailed split information per domain
    """
    total_docs = sum(len(docs) for docs in documents_by_domain.values())
    assert train_size + dev_size + test_size == total_docs, \
        f"Split sizes ({train_size}+{dev_size}+{test_size}) must equal total docs ({total_docs})"
    
    random.seed(random_seed)
    
    train_docs = []
    dev_docs = []
    test_docs = []
    split_info = {}
    
    print("\n" + "="*70)
    print("STRATIFIED SPLIT BY DOMAIN")
    print("="*70)
    print(f"{'Domain':<12} {'Total':>5} {'Train':>5} {'Dev':>5} {'Test':>5} {'Train%':>7} {'Dev%':>7} {'Test%':>7}")
    print("-"*70)
    
    for domain, docs in documents_by_domain.items():
        n_docs = len(docs)
        
        # Calculate proportional split for this domain
        n_train = round(n_docs * train_size / total_docs)
        n_dev = round(n_docs * dev_size / total_docs)
        n_test = n_docs - n_train - n_dev  # remaining goes to test
        
        # Shuffle documents with fixed seed
        shuffled_docs = docs.copy()
        random.shuffle(shuffled_docs)
        
        # Split
        domain_train = shuffled_docs[:n_train]
        domain_dev = shuffled_docs[n_train:n_train+n_dev]
        domain_test = shuffled_docs[n_train+n_dev:]
        
        # Add domain label to each doc
        train_docs.extend([(doc_id, doc_text, domain) for doc_id, doc_text in domain_train])
        dev_docs.extend([(doc_id, doc_text, domain) for doc_id, doc_text in domain_dev])
        test_docs.extend([(doc_id, doc_text, domain) for doc_id, doc_text in domain_test])
        
        # Store split info
        split_info[domain] = {
            'total': n_docs,
            'train': n_train,
            'dev': n_dev,
            'test': n_test,
            'train_ids': [doc_id for doc_id, _ in domain_train],
            'dev_ids': [doc_id for doc_id, _ in domain_dev],
            'test_ids': [doc_id for doc_id, _ in domain_test]
        }
        
        # Print with percentages
        train_pct = f"{100*n_train/n_docs:.1f}%"
        dev_pct = f"{100*n_dev/n_docs:.1f}%"
        test_pct = f"{100*n_test/n_docs:.1f}%"
        
        print(f"{domain:<12} {n_docs:>5} {n_train:>5} {n_dev:>5} {n_test:>5} {train_pct:>7} {dev_pct:>7} {test_pct:>7}")
    
    print("-"*70)
    print(f"{'TOTAL':<12} {total_docs:>5} {len(train_docs):>5} {len(dev_docs):>5} {len(test_docs):>5}")
    print("="*70)
    
    return train_docs, dev_docs, test_docs, split_info

print("✓ Function 'stratified_split' defined")

✓ Function 'stratified_split' defined


In [6]:
# Split BenCoref: 41 train, 10 dev, 71 test (all stratified by domain)
print("Performing stratified split on BenCoref (122 documents)...")
print(f"Target: 41 train, 10 dev, 71 test")

bencoref_train, bencoref_dev, bencoref_test, split_info = stratified_split(
    bencoref_by_domain,
    train_size=41,
    dev_size=10,
    test_size=71,
    random_seed=RANDOM_SEED
)

print(f"\n✓ Split complete!")
print(f"  Train: {len(bencoref_train)} BenCoref docs (stratified)")
print(f"  Dev:   {len(bencoref_dev)} BenCoref docs (stratified)")
print(f"  Test:  {len(bencoref_test)} BenCoref docs (stratified)")

Performing stratified split on BenCoref (122 documents)...
Target: 41 train, 10 dev, 71 test

STRATIFIED SPLIT BY DOMAIN
Domain       Total Train   Dev  Test  Train%    Dev%   Test%
----------------------------------------------------------------------
biography       17     6     1    10   35.3%    5.9%   58.8%
descriptive     36    12     3    21   33.3%    8.3%   58.3%
novel           13     4     1     8   30.8%    7.7%   61.5%
story           56    19     5    32   33.9%    8.9%   57.1%
----------------------------------------------------------------------
TOTAL          122    41    10    71

✓ Split complete!
  Train: 41 BenCoref docs (stratified)
  Dev:   10 BenCoref docs (stratified)
  Test:  71 BenCoref docs (stratified)


In [7]:
# Load TransMuCoRes (100 Bengali documents)
print("Loading TransMuCoRes documents...")
print("="*60)

transmucores_docs = extract_documents_from_conll(TRANSMUCORES_FILE)
print(f"✓ Loaded {len(transmucores_docs)} TransMuCoRes documents")

# Verify count
if len(transmucores_docs) == 100:
    print("✓ Correct! TransMuCoRes has 100 documents")
else:
    print(f"⚠️  Warning: Expected 100 documents, got {len(transmucores_docs)}")

# Add domain label for TransMuCoRes (use 'transmucores' as pseudo-domain)
transmucores_train = [(doc_id, doc_text, 'transmucores') 
                      for doc_id, doc_text in transmucores_docs]

print(f"\n✓ Prepared {len(transmucores_train)} TransMuCoRes docs for training")
print("="*60)

Loading TransMuCoRes documents...
✓ Loaded 100 TransMuCoRes documents
✓ Correct! TransMuCoRes has 100 documents

✓ Prepared 100 TransMuCoRes docs for training


In [10]:
# Create all 6 experiment configurations
print("="*70)
print("ASSEMBLING EXPERIMENT CONFIGURATIONS")
print("="*70)

# Experiment 1: TransMuCoRes + BenCoref
exp1_train = transmucores_train + bencoref_train
exp1_dev = bencoref_dev
exp1_test = bencoref_test

print("\n✓ Experiment 1: Mixed Bengali Datasets")
print(f"  Train: {len(exp1_train)} docs (100 TransMuCoRes + 41 BenCoref)")
print(f"  Dev:   {len(exp1_dev)} docs (10 BenCoref)")
print(f"  Test:  {len(exp1_test)} docs (71 BenCoref)")

# Experiment 2: BenCoref only (baseline)
exp2_train = bencoref_train
exp2_dev = bencoref_dev
exp2_test = bencoref_test

print("\n✓ Experiment 2: BenCoref Only (Baseline)")
print(f"  Train: {len(exp2_train)} docs (41 BenCoref)")
print(f"  Dev:   {len(exp2_dev)} docs (10 BenCoref)")
print(f"  Test:  {len(exp2_test)} docs (71 BenCoref)")

# Experiment 3: BenCoref + Back-translated BenCoref
print("\n⚠️  Experiment 3: Back-translated BenCoref Augmentation")
print(f"  Train: Will be 82 docs (41 BenCoref + 41 Back-translated BenCoref)")
print(f"  Dev:   {len(bencoref_dev)} docs (10 BenCoref)")
print(f"  Test:  {len(bencoref_test)} docs (71 BenCoref)")
print(f"  Status: Awaiting augmented data")

# Experiment 4: BenCoref + Paraphrased BenCoref
print("\n⚠️  Experiment 4: Paraphrased BenCoref Augmentation")
print(f"  Train: Will be 82 docs (41 BenCoref + 41 Paraphrased BenCoref)")
print(f"  Dev:   {len(bencoref_dev)} docs (10 BenCoref)")
print(f"  Test:  {len(bencoref_test)} docs (71 BenCoref)")
print(f"  Status: Awaiting augmented data")

# Experiment 5: Full dataset + Back-translated Full dataset
print("\n⚠️  Experiment 5: Back-translated Full Dataset Augmentation")
print(f"  Train: Will be 282 docs (100 TransMuCoRes + 41 BenCoref + 141 Back-translated)")
print(f"  Dev:   {len(bencoref_dev)} docs (10 BenCoref)")
print(f"  Test:  {len(bencoref_test)} docs (71 BenCoref)")
print(f"  Status: Awaiting augmented data")

# Experiment 6: Full dataset + Paraphrased Full dataset
print("\n⚠️  Experiment 6: Paraphrased Full Dataset Augmentation")
print(f"  Train: Will be 282 docs (100 TransMuCoRes + 41 BenCoref + 141 Paraphrased)")
print(f"  Dev:   {len(bencoref_dev)} docs (10 BenCoref)")
print(f"  Test:  {len(bencoref_test)} docs (71 BenCoref)")
print(f"  Status: Awaiting augmented data")

print("\n" + "="*70)
print("KEY POINTS:")
print("="*70)
print("✓ Same DEV set (10 docs) across ALL 6 experiments")
print("✓ Same TEST set (71 docs) across ALL 6 experiments")
print("✓ All splits are stratified by domain")
print("="*70)

ASSEMBLING EXPERIMENT CONFIGURATIONS

✓ Experiment 1: Mixed Bengali Datasets
  Train: 141 docs (100 TransMuCoRes + 41 BenCoref)
  Dev:   10 docs (10 BenCoref)
  Test:  71 docs (71 BenCoref)

✓ Experiment 2: BenCoref Only (Baseline)
  Train: 41 docs (41 BenCoref)
  Dev:   10 docs (10 BenCoref)
  Test:  71 docs (71 BenCoref)

⚠️  Experiment 3: Back-translated BenCoref Augmentation
  Train: Will be 82 docs (41 BenCoref + 41 Back-translated BenCoref)
  Dev:   10 docs (10 BenCoref)
  Test:  71 docs (71 BenCoref)
  Status: Awaiting augmented data

⚠️  Experiment 4: Paraphrased BenCoref Augmentation
  Train: Will be 82 docs (41 BenCoref + 41 Paraphrased BenCoref)
  Dev:   10 docs (10 BenCoref)
  Test:  71 docs (71 BenCoref)
  Status: Awaiting augmented data

⚠️  Experiment 5: Back-translated Full Dataset Augmentation
  Train: Will be 282 docs (100 TransMuCoRes + 41 BenCoref + 141 Back-translated)
  Dev:   10 docs (10 BenCoref)
  Test:  71 docs (71 BenCoref)
  Status: Awaiting augmented data



In [13]:
def write_conll_file(documents, filepath):
    """
    Write documents to a CoNLL file.
    
    Args:
        documents: list of (doc_id, doc_text, domain) tuples
        filepath: output file path
    """
    with open(filepath, 'w', encoding='utf-8') as f:
        for i, (doc_id, doc_text, domain) in enumerate(documents):
            f.write(doc_text)
            if i < len(documents) - 1:  # Add blank line between documents
                f.write('\n\n')
            else:
                f.write('\n')
    
    return len(documents)

def count_by_domain(docs):
    """Count documents by domain."""
    counts = defaultdict(int)
    for _, _, domain in docs:
        counts[domain] += 1
    return dict(counts)

print("✓ Functions 'write_conll_file' and 'count_by_domain' defined")

✓ Functions 'write_conll_file' and 'count_by_domain' defined


In [14]:
print("="*70)
print("WRITING DATASETS TO DISK")
print("="*70)

# Create experiment directories
exp1_dir = OUTPUT_DIR / 'experiment_1_transmucores_bencoref'
exp2_dir = OUTPUT_DIR / 'experiment_2_bencoref_only'

exp1_dir.mkdir(exist_ok=True)
exp2_dir.mkdir(exist_ok=True)

# ============================================================
# EXPERIMENT 1: TransMuCoRes + BenCoref
# ============================================================
print("\n📁 Experiment 1: Mixed Bengali Datasets")
print(f"   Directory: {exp1_dir.name}")

exp1_train_file = exp1_dir / 'train.conll'
exp1_dev_file = exp1_dir / 'dev.conll'
exp1_test_file = exp1_dir / 'test.conll'

write_conll_file(exp1_train, exp1_train_file)
write_conll_file(exp1_dev, exp1_dev_file)
write_conll_file(exp1_test, exp1_test_file)

print(f"   ✓ train.conll: {len(exp1_train)} docs")
print(f"   ✓ dev.conll:   {len(exp1_dev)} docs")
print(f"   ✓ test.conll:  {len(exp1_test)} docs")

# Show domain composition
train_composition = count_by_domain(exp1_train)
print(f"   Train composition: {dict(sorted(train_composition.items()))}")

# ============================================================
# EXPERIMENT 2: BenCoref Only (Baseline)
# ============================================================
print("\n📁 Experiment 2: BenCoref Only (Baseline)")
print(f"   Directory: {exp2_dir.name}")

exp2_train_file = exp2_dir / 'train.conll'
exp2_dev_file = exp2_dir / 'dev.conll'
exp2_test_file = exp2_dir / 'test.conll'

write_conll_file(exp2_train, exp2_train_file)
write_conll_file(exp2_dev, exp2_dev_file)
write_conll_file(exp2_test, exp2_test_file)

print(f"   ✓ train.conll: {len(exp2_train)} docs")
print(f"   ✓ dev.conll:   {len(exp2_dev)} docs")
print(f"   ✓ test.conll:  {len(exp2_test)} docs")

# Show domain composition
train_composition = count_by_domain(exp2_train)
print(f"   Train composition: {dict(sorted(train_composition.items()))}")

print("\n" + "="*70)
print("✓ Experiments 1 & 2 datasets written successfully!")
print("="*70)

WRITING DATASETS TO DISK

📁 Experiment 1: Mixed Bengali Datasets
   Directory: experiment_1_transmucores_bencoref
   ✓ train.conll: 141 docs
   ✓ dev.conll:   10 docs
   ✓ test.conll:  71 docs
   Train composition: {'biography': 6, 'descriptive': 12, 'novel': 4, 'story': 19, 'transmucores': 100}

📁 Experiment 2: BenCoref Only (Baseline)
   Directory: experiment_2_bencoref_only
   ✓ train.conll: 41 docs
   ✓ dev.conll:   10 docs
   ✓ test.conll:  71 docs
   Train composition: {'biography': 6, 'descriptive': 12, 'novel': 4, 'story': 19}

✓ Experiments 1 & 2 datasets written successfully!


In [15]:
# Prepare complete split information for reproducibility
complete_split_info = {
    'random_seed': RANDOM_SEED,
    'creation_date': '2024-12-09',
    
    # Overall dataset statistics
    'bencoref_total': 122,
    'transmucores_total': 100,
    
    # BenCoref stratification details
    'bencoref_stratification': split_info,
    
    # Experiment configurations
    'experiments': {
        'experiment_1': {
            'name': 'Mixed Bengali Datasets (TransMuCoRes + BenCoref)',
            'train': {
                'total': len(exp1_train),
                'transmucores': 100,
                'bencoref': 41
            },
            'dev': {'total': len(exp1_dev), 'bencoref': 10},
            'test': {'total': len(exp1_test), 'bencoref': 71},
            'status': 'complete'
        },
        'experiment_2': {
            'name': 'BenCoref Only (Baseline)',
            'train': {'total': len(exp2_train), 'bencoref': 41},
            'dev': {'total': len(exp2_dev), 'bencoref': 10},
            'test': {'total': len(exp2_test), 'bencoref': 71},
            'status': 'complete'
        },
        'experiment_3': {
            'name': 'Back-translated BenCoref Augmentation',
            'train': {
                'total': 82,
                'bencoref': 41,
                'backtranslated_bencoref': 41
            },
            'dev': {'total': 10, 'bencoref': 10},
            'test': {'total': 71, 'bencoref': 71},
            'status': 'awaiting_augmented_data',
            'note': 'Back-translate the 41 BenCoref training docs'
        },
        'experiment_4': {
            'name': 'Paraphrased BenCoref Augmentation',
            'train': {
                'total': 82,
                'bencoref': 41,
                'paraphrased_bencoref': 41
            },
            'dev': {'total': 10, 'bencoref': 10},
            'test': {'total': 71, 'bencoref': 71},
            'status': 'awaiting_augmented_data',
            'note': 'Paraphrase the 41 BenCoref training docs'
        },
        'experiment_5': {
            'name': 'Back-translated Full Dataset Augmentation',
            'train': {
                'total': 282,
                'transmucores': 100,
                'bencoref': 41,
                'backtranslated': 141
            },
            'dev': {'total': 10, 'bencoref': 10},
            'test': {'total': 71, 'bencoref': 71},
            'status': 'awaiting_augmented_data',
            'note': 'Back-translate 100 TransMuCoRes + 41 BenCoref training docs'
        },
        'experiment_6': {
            'name': 'Paraphrased Full Dataset Augmentation',
            'train': {
                'total': 282,
                'transmucores': 100,
                'bencoref': 41,
                'paraphrased': 141
            },
            'dev': {'total': 10, 'bencoref': 10},
            'test': {'total': 71, 'bencoref': 71},
            'status': 'awaiting_augmented_data',
            'note': 'Paraphrase 100 TransMuCoRes + 41 BenCoref training docs'
        }
    },
    
    # Document IDs for reproducibility
    'document_ids': {
        'shared_across_all_experiments': {
            'train_bencoref_41': [doc_id for doc_id, _, _ in bencoref_train],
            'dev_10': [doc_id for doc_id, _, _ in bencoref_dev],
            'test_71': [doc_id for doc_id, _, _ in bencoref_test]
        },
        'experiment_1_5_6': {
            'train_transmucores_100': [doc_id for doc_id, _, _ in transmucores_train]
        }
    },
    
    # Summary
    'summary': {
        'total_experiments': 6,
        'completed': 2,
        'awaiting_augmentation': 4,
        'common_dev_test': 'All experiments share the same dev (10 docs) and test (71 docs) sets'
    }
}

# Save to JSON
split_info_file = OUTPUT_DIR / 'dataset_split_info.json'
with open(split_info_file, 'w', encoding='utf-8') as f:
    json.dump(complete_split_info, f, indent=2, ensure_ascii=False)

print("="*70)
print("✓ Complete split information saved!")
print("="*70)
print(f"\nFile: {split_info_file}")
print(f"\nThis file contains:")
print(f"  • Random seed: {RANDOM_SEED}")
print(f"  • All 6 experiment configurations")
print(f"  • Domain stratification details")
print(f"  • Document IDs for each split")
print(f"  • Status of each experiment")
print("\n" + "="*70)

✓ Complete split information saved!

File: /teamspace/studios/this_studio/final_experiments/dataset_split_info.json

This file contains:
  • Random seed: 42
  • All 6 experiment configurations
  • Domain stratification details
  • Document IDs for each split
  • Status of each experiment



In [16]:
print("="*70)
print("DATASET CREATION SUMMARY")
print("="*70)

print("\n✅ COMPLETED (2/6 experiments):")
print("-"*70)

print("\n📁 Experiment 1: experiment_1_transmucores_bencoref/")
print(f"   • train.conll: 141 docs (100 TransMuCoRes + 41 BenCoref)")
print(f"   • dev.conll:   10 docs")
print(f"   • test.conll:  71 docs")

print("\n📁 Experiment 2: experiment_2_bencoref_only/")
print(f"   • train.conll: 41 docs (BenCoref only)")
print(f"   • dev.conll:   10 docs")
print(f"   • test.conll:  71 docs")

print("\n" + "="*70)
print("⚠️  AWAITING AUGMENTED DATA (4/6 experiments):")
print("="*70)

print("\n📋 Experiment 3: Back-translated BenCoref")
print(f"   Need: 41 back-translated versions of BenCoref train docs")
print(f"   Final train size: 82 docs")

print("\n📋 Experiment 4: Paraphrased BenCoref")
print(f"   Need: 41 paraphrased versions of BenCoref train docs")
print(f"   Final train size: 82 docs")

print("\n📋 Experiment 5: Back-translated Full Dataset")
print(f"   Need: 141 back-translated versions (100 TransMuCoRes + 41 BenCoref)")
print(f"   Final train size: 282 docs")

print("\n📋 Experiment 6: Paraphrased Full Dataset")
print(f"   Need: 141 paraphrased versions (100 TransMuCoRes + 41 BenCoref)")
print(f"   Final train size: 282 docs")

print("\n" + "="*70)
print("📊 KEY STATISTICS:")
print("="*70)
print(f"Random Seed: {RANDOM_SEED}")
print(f"\nShared across ALL experiments:")
print(f"  • Dev set:  {len(bencoref_dev)} docs (stratified BenCoref)")
print(f"  • Test set: {len(bencoref_test)} docs (stratified BenCoref)")

print(f"\nDomain distribution in test set:")
test_domains = count_by_domain(bencoref_test)
for domain, count in sorted(test_domains.items()):
    print(f"  • {domain:12s}: {count:2d} docs")

print("\n" + "="*70)
print("📁 OUTPUT STRUCTURE:")
print("="*70)
print(f"{OUTPUT_DIR}/")
print("├── experiment_1_transmucores_bencoref/")
print("│   ├── train.conll (141 docs) ✅")
print("│   ├── dev.conll (10 docs) ✅")
print("│   └── test.conll (71 docs) ✅")
print("├── experiment_2_bencoref_only/")
print("│   ├── train.conll (41 docs) ✅")
print("│   ├── dev.conll (10 docs) ✅")
print("│   └── test.conll (71 docs) ✅")
print("└── dataset_split_info.json ✅")

print("\n" + "="*70)
print("🚀 NEXT STEPS:")
print("="*70)
print("1. Create augmented data for Experiments 3-6:")
print("   • Back-translate 41 BenCoref train docs (for Exp 3)")
print("   • Paraphrase 41 BenCoref train docs (for Exp 4)")
print("   • Back-translate 141 full train docs (for Exp 5)")
print("   • Paraphrase 141 full train docs (for Exp 6)")
print("\n2. Once augmented data is ready, create final datasets")
print("\n3. Use separate notebook for training & evaluation")

print("\n" + "="*70)
print("✅ Base dataset creation complete!")
print("="*70)

DATASET CREATION SUMMARY

✅ COMPLETED (2/6 experiments):
----------------------------------------------------------------------

📁 Experiment 1: experiment_1_transmucores_bencoref/
   • train.conll: 141 docs (100 TransMuCoRes + 41 BenCoref)
   • dev.conll:   10 docs
   • test.conll:  71 docs

📁 Experiment 2: experiment_2_bencoref_only/
   • train.conll: 41 docs (BenCoref only)
   • dev.conll:   10 docs
   • test.conll:  71 docs

⚠️  AWAITING AUGMENTED DATA (4/6 experiments):

📋 Experiment 3: Back-translated BenCoref
   Need: 41 back-translated versions of BenCoref train docs
   Final train size: 82 docs

📋 Experiment 4: Paraphrased BenCoref
   Need: 41 paraphrased versions of BenCoref train docs
   Final train size: 82 docs

📋 Experiment 5: Back-translated Full Dataset
   Need: 141 back-translated versions (100 TransMuCoRes + 41 BenCoref)
   Final train size: 282 docs

📋 Experiment 6: Paraphrased Full Dataset
   Need: 141 paraphrased versions (100 TransMuCoRes + 41 BenCoref)
   Final t